In [1]:
from torch import nn as nn
import torch
import torch.autograd as ag
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def differential_equation(t:torch.Tensor):
    r: float = 1.0
    return r*t*(1-t)

def solution(x):
    return (-1/3)*x**3 + (1/2)*x**2 + 1

In [ ]:
#Building Neural Network Model
#base class for all neural network modules
class PINNModel(nn.Module):
    def __init__(self, input_size: int = 1, output_size: int = 1, hidden_layers: int = 5, hidden_nodes: int = 10):
        super().__init__()
        #connecting the layers using nn.Linear(input,output)
        self.inputs = nn.Linear(input_size, hidden_nodes)
        self.model = nn.ModuleList([nn.Linear(hidden_nodes, hidden_nodes) for _ in range(hidden_layers)])
        self.outputs = nn.Linear(hidden_nodes, output_size)

        self.activation = nn.Tanh() # makes it nonlinear

    #applies activation to all layers in the model
    def _apply_model(self,x: torch.Tensor):
        for layer in self.model:
            x = self.activation(layer(x))
        return x

    def forward(self, t: torch.Tensor):
        t = self.activation(self.inputs(t))
        t = self._apply_model(t)
        t = self.outputs(t)
        return t


In [ ]:
#Preparing Points for Training
x = torch.linspace(0, 1, 100).reshape(-1, 1) #many data points
y = solution(x).reshape(-1, 1) #analytical solution for the differential equation

#take only certain points for training
x_data = x[0:200:20]
y_data = y[0:200:20]

#convert form numpy to tensor
x_data_t = torch.tensor(x_data, dtype=torch.float32, device="cuda").reshape(-1, 1)
y_data_t = torch.tensor(y_data, dtype=torch.float32, device="cuda").reshape(-1, 1)


#creating points for physics loss
x_physics = torch.linspace(0, 1, 100, requires_grad=True, device="cuda", dtype=torch.float32).reshape(-1, 1)



In [ ]:
#Training Loop
#Goals: Match the data points and satisfy the differential equation
Epochs = 10000
Length = 1e4

model = PINNModel().to("cuda")
optimize = torch.optim.Adam(model.parameters(), lr=1e-3)

data_loss = nn.MSELoss()

for epoch in range(Epochs):
    optimize.zero_grad()
    
    #data loss
    y_pred = model(x_data_t)
    loss_data = data_loss(y_pred, y_data_t)

    #physics loss
    y_physics = model(x_physics)
    dy_physics = ag.grad(y_physics, x_physics, grad_outputs=torch.ones
    total_loss = loss_data + loss_physics

    total_loss.backward()
    optimize.step()


